# Aeolus Williamson-5 MRI projected-reference validation (Notebook B)

This notebook runs canonical Aeolus Williamson test case 5 at T42 on 64×128 and T63 on 96×192, then compares the saved solutions with independently prepared, hash-pinned MRI→T42 and MRI→T63 projections.

**Reference firewall:** Notebook B never reads the original MRI `data.nc`, installs/imports Skyborn, invokes SPHEREPACK, performs MRI spectral analysis/projection/wind reconstruction, interpolates/regrids fields, adds times, or repairs/regenerates reference artifacts. It consumes exactly `manifest.json`, `t42_64x128.npz`, and `t63_96x192.npz` relative to `REFERENCE_DIR` after verifying all three SHA-256 hashes.

The workflow reports four distinct views: **day-zero convention verification**, **MRI projected-reference agreement**, **Aeolus conservation and stability**, and **T42–T63 self-convergence**. Passing a project-defined check is not claimed to be an official Williamson pass.

In [ ]:
# ---- User configuration (edit only this cell) -----------------------------
REPO_URL = "https://github.com/AlexandreEros/Aeolus.git"
GIT_REF = "feat/w5-mri-validation"
DRIVE_ROOT = "/content/drive/MyDrive"
REFERENCE_DIR = f"{DRIVE_ROOT}/mri-w5-reference-v2"
OUTPUT_ROOT = f"{DRIVE_ROOT}/w5-mri-validation-v2"
RUN_T42 = True
RUN_T63 = True
FORCE_RERUN = False
REUSE_T42_CAPSULE = None  # set to a completed capsule to post-process only
REUSE_T63_CAPSULE = None

# Disposable Colab checkout; scientific outputs remain under OUTPUT_ROOT.
REPO_DIR = "/content/Aeolus"

print({
    "REPO_URL": REPO_URL,
    "GIT_REF": GIT_REF,
    "DRIVE_ROOT": DRIVE_ROOT,
    "REFERENCE_DIR": REFERENCE_DIR,
    "OUTPUT_ROOT": OUTPUT_ROOT,
    "RUN_T42": RUN_T42,
    "RUN_T63": RUN_T63,
    "FORCE_RERUN": FORCE_RERUN,
    "REUSE_T42_CAPSULE": REUSE_T42_CAPSULE,
    "REUSE_T63_CAPSULE": REUSE_T63_CAPSULE,
})

## Mount Drive and report the GPU

T42 and T63 are separate resumable stages. A stage is reusable only when its signature and every hash in its atomically written `COMPLETE.json` still match.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import pathlib
import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)
print("GPU:", gpu.stdout.strip() or f"not detected (return code {gpu.returncode})")
if gpu.returncode != 0:
    raise RuntimeError("A CUDA GPU runtime is required for the canonical Aeolus runs")

reference_path = pathlib.Path(REFERENCE_DIR)
if not reference_path.is_dir():
    raise FileNotFoundError(f"REFERENCE_DIR does not exist: {reference_path}")

## Reproducible Aeolus checkout

An existing Colab checkout is updated only if clean. The cell fetches `GIT_REF`, checks out the fetched commit in detached mode, and reports the exact SHA and dirty status. It never resets, cleans, stashes, or discards a dirty checkout.

In [ ]:
repo = pathlib.Path(REPO_DIR)

def git(*args, check=True):
    return subprocess.run(
        ["git", "-C", str(repo), *args], text=True,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=check)

if (repo / ".git").is_dir():
    existing_status = git("status", "--porcelain").stdout
    if existing_status:
        raise RuntimeError(
            "Existing Colab checkout is dirty; refusing to switch refs or discard work:\n"
            + existing_status)
    git("remote", "set-url", "origin", REPO_URL)
else:
    subprocess.run(
        ["git", "clone", "--no-checkout", REPO_URL, str(repo)], check=True)

git("fetch", "--no-tags", "origin", GIT_REF)
git("checkout", "--detach", "FETCH_HEAD")
commit_sha = git("rev-parse", "HEAD").stdout.strip()
dirty_status = git("status", "--porcelain").stdout
print("Aeolus commit SHA:", commit_sha)
print("Dirty tree:", bool(dirty_status))
if dirty_status:
    print(dirty_status)

## Install Aeolus dependencies

Only the repository's declared dependencies are installed. Notebook A's external spherical-harmonic stack is intentionally absent.

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
    cwd=repo, check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."],
    cwd=repo, check=True)

import cupy as cp
print("CuPy:", cp.__version__)
print("CUDA runtime:", cp.cuda.runtime.runtimeGetVersion())
print("Skyborn installed/imported by Notebook B: False")

## Verify the frozen package before any Aeolus run

File discovery is relative to `REFERENCE_DIR`; absolute Windows paths retained in the preparation manifest are provenance only.

In [ ]:
from planetary_sandbox.validation.w5_mri import (
    derive_day0_tolerances, verify_reference_package)

reference = verify_reference_package(REFERENCE_DIR)
day0_tolerances = derive_day0_tolerances(reference.manifest)
print("Reference schema:", reference.manifest["schema_version"])
print("Verified SHA-256:")
for name, digest in reference.hashes.items():
    print(f"  {name}: {digest}")
print("Predeclared day-zero tolerances:", day0_tolerances)

## Execute or resume T42 and T63

The audited Aeolus state convention is:

```text
layer depth            = (Phi0 + phi) / gravity
free-surface elevation = (Phi0 + phi + phi_s) / gravity
```

`phi` is perturbation fluid-layer geopotential and `Phi0=gH` is restored exactly once. MRI `h` is compared only with layer depth; fixed `phi_s` is used only for a separate optional free-surface-elevation diagnostic. The day-zero layer-depth gate is evaluated and persisted before any integration callback can run, then checked again against the saved day-zero snapshot.

In [ ]:
from planetary_sandbox.validation.w5_mri_workflow import (
    WorkflowConfig, run_validation_workflow)

result = run_validation_workflow(WorkflowConfig(
    reference_dir=pathlib.Path(REFERENCE_DIR),
    output_root=pathlib.Path(OUTPUT_ROOT),
    repository_root=repo,
    repository_url=REPO_URL,
    git_ref=GIT_REF,
    run_t42=RUN_T42,
    run_t63=RUN_T63,
    force_rerun=FORCE_RERUN,
    reuse_t42_capsule=(
        None if REUSE_T42_CAPSULE is None else pathlib.Path(REUSE_T42_CAPSULE)
    ),
    reuse_t63_capsule=(
        None if REUSE_T63_CAPSULE is None else pathlib.Path(REUSE_T63_CAPSULE)
    ),
))
result

## Final artifact inventory

The validation manifest is strict JSON with no NaN or infinity. Normalized metrics with a mathematically zero reference denominator are represented as `null` with an explanatory status.

In [ ]:
from planetary_sandbox.validation.w5_mri import sha256_file

output = pathlib.Path(OUTPUT_ROOT)
important = [output / "validation_manifest.json"]
important.extend(sorted(output.glob("t*_*/COMPLETE.json")))
important.extend([
    output / "summary" / "metrics_table.csv",
    output / "summary" / "README.md",
])
print("Final artifacts:")
for path in important:
    if path.is_file():
        print(f"  {path}  sha256={sha256_file(path)}")